
# 36 蒙特卡洛模拟：从路径到方差缩减

> **本章要解决的问题**：期权价、VaR、组合分布，解析公式算不出来时怎么办？为什么“多跑几条路径”不等于“更准”？同样 1 万条路径，怎样把标准误降一半？

## 本章导读

第32章用蒙特卡洛算过障碍期权；这一章把它系统化：

1. 随机数：逆 CDF、可复现；
2. 路径：GBM/OU/跳扩散、离散误差；
3. 误差：标准误、置信区间、收敛速度；
4. 方差缩减：对偶、控制变量、分层、重要性抽样、拟蒙特卡洛；
5. 应用：期权、VaR/ES、组合分布。

### 学习目标

- 能用逆 CDF 从均匀随机数生成正态/厚尾样本，并固定种子复现。
- 能模拟 GBM/OU/跳扩散路径，并解释离散误差。
- 能算蒙特卡洛均值的标准误和 95% 置信区间。
- 能用对偶/控制变量/分层把标准误降下来。
- 能用蒙特卡洛算期权价、VaR、ES。
- 能判断“路径数够不够”，而不是只看结果像不像。

### 先修

第02章的正态/期望/方差；第06章的布朗运动/GBM；会 numpy。

---

### 环境准备



## 本章知识结构图

<svg style="max-width:100%;height:auto" xmlns="http://www.w3.org/2000/svg" width="800" height="450" viewBox="0 0 800 450" role="img" aria-label="36 蒙特卡洛模拟：从路径到方差缩减" font-family="Microsoft YaHei, SimHei, sans-serif">
<title>36 蒙特卡洛模拟：从路径到方差缩减｜知识图</title>
<desc>本章主线：路径均值只是一半，报出标准误和区间；对偶、控制变量把方差压下去。</desc>
<rect x="0" y="0" width="800" height="450" fill="#FFFFFF"/>
<defs><marker id="a36" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#1B3A5C"/></marker><marker id="ao36" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#E07B39"/></marker><marker id="ag36" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#6E7B8B"/></marker></defs>
<rect x="40" y="19" width="46" height="29" fill="#E07B39" rx="8"/>
<text x="63" y="39" font-size="15" fill="#FFFFFF" text-anchor="middle" font-weight="bold">36</text>
<text x="98" y="41" font-size="22" fill="#1B3A5C" font-weight="bold">蒙特卡洛模拟：从路径到方差缩减</text>
<line x1="40" y1="57" x2="760" y2="57" stroke="#E6EFF7" stroke-width="2" stroke-linecap="round"/>
<line x1="40" y1="407" x2="760" y2="407" stroke="#E6EFF7" stroke-width="1.6" stroke-linecap="round"/>
<rect x="40" y="418" width="4" height="10" fill="#E07B39" rx="1"/>
<text x="52" y="428" font-size="13" fill="#6E7B8B">本章主线：路径均值只是一半，报出标准误和区间；对偶、控制变量把方差压下去。</text>
<path d="M 75 300 L 585 300" fill="none" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag36)"/>
<path d="M 75 300 L 75 300" fill="none" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag36)"/>
<path d="M 75 300 L 75 300" fill="none" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag36)"/>
<path d="M 75 300 L 75 108" fill="none" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag36)"/>
<text x="82" y="120" font-size="9.5" fill="#6E7B8B">价格路径</text>
<text x="566" y="318" font-size="9.5" fill="#6E7B8B">t</text>
<path d="M 85 225.3 L 101 218.1 L 117 212.3 L 133 207.3 L 149 202.7 L 165 198.5 L 181 194.5 L 197 190.7 L 213 187.1 L 229 183.6 L 245 180.2 L 261 177 L 277 173.8 L 293 170.7 L 309 167.7 L 325 164.7 L 341 161.8 L 357 158.9 L 373 156.1 L 389 153.4 L 405 150.6 L 421 148 L 437 145.3 L 453 142.7 L 469 140.1 L 485 137.5 L 501 135 L 517 132.5 L 533 130 L 549 127.6 L 565 125.2 L 565 288.8 L 549 288.6 L 533 288.4 L 517 288.1 L 501 287.8 L 485 287.5 L 469 287.1 L 453 286.7 L 437 286.3 L 421 285.8 L 405 285.4 L 389 284.8 L 373 284.3 L 357 283.7 L 341 283 L 325 282.3 L 309 281.5 L 293 280.7 L 277 279.8 L 261 278.8 L 245 277.8 L 229 276.6 L 213 275.3 L 197 273.9 L 181 272.3 L 165 270.5 L 149 268.5 L 133 266.1 L 117 263.3 L 101 259.7 L 85 254.7" fill="#E6EFF7" stroke="#E6EFF7" stroke-width="0.3" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M 85 225.3 L 101 218.1 L 117 212.3 L 133 207.3 L 149 202.7 L 165 198.5 L 181 194.5 L 197 190.7 L 213 187.1 L 229 183.6 L 245 180.2 L 261 177 L 277 173.8 L 293 170.7 L 309 167.7 L 325 164.7 L 341 161.8 L 357 158.9 L 373 156.1 L 389 153.4 L 405 150.6 L 421 148 L 437 145.3 L 453 142.7 L 469 140.1 L 485 137.5 L 501 135 L 517 132.5 L 533 130 L 549 127.6 L 565 125.2" fill="none" stroke="#E07B39" stroke-width="1.2" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="5 4"/>
<path d="M 85 254.7 L 101 259.7 L 117 263.3 L 133 266.1 L 149 268.5 L 165 270.5 L 181 272.3 L 197 273.9 L 213 275.3 L 229 276.6 L 245 277.8 L 261 278.8 L 277 279.8 L 293 280.7 L 309 281.5 L 325 282.3 L 341 283 L 357 283.7 L 373 284.3 L 389 284.8 L 405 285.4 L 421 285.8 L 437 286.3 L 453 286.7 L 469 287.1 L 485 287.5 L 501 287.8 L 517 288.1 L 533 288.4 L 549 288.6 L 565 288.8" fill="none" stroke="#E07B39" stroke-width="1.2" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="5 4"/>
<path d="M 85 240 L 101 238.7 L 117 233.8 L 133 230.4 L 149 243 L 165 234.6 L 181 234.4 L 197 240.5 L 213 220.8 L 229 245.4 L 245 222.4 L 261 238.2 L 277 232.1 L 293 261.3 L 309 231.2 L 325 231.6 L 341 222.4 L 357 199.7 L 373 213.4 L 389 233.4 L 405 193.4 L 421 226.7 L 437 224.7 L 453 212.6 L 469 221.6 L 485 214.4 L 501 196 L 517 189.1 L 533 208.7 L 549 186.6 L 565 203.4" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 225.9 L 117 234.6 L 133 239.7 L 149 242.8 L 165 235.7 L 181 238.4 L 197 232.8 L 213 233.9 L 229 242.8 L 245 224.9 L 261 225.5 L 277 212.2 L 293 230.3 L 309 213.3 L 325 216.9 L 341 228.7 L 357 234.5 L 373 194.7 L 389 245.9 L 405 215.7 L 421 243.4 L 437 217.9 L 453 194 L 469 209 L 485 216.3 L 501 170.9 L 517 205.5 L 533 211.2 L 549 191 L 565 206.2" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 231.6 L 117 241.7 L 133 235.1 L 149 226.3 L 165 259.8 L 181 234.2 L 197 231.2 L 213 254 L 229 234.1 L 245 230.8 L 261 224.6 L 277 215.6 L 293 210.8 L 309 262.7 L 325 224.4 L 341 237.6 L 357 227.4 L 373 230.9 L 389 208.6 L 405 193.8 L 421 212.2 L 437 210.6 L 453 197.7 L 469 201.9 L 485 208.8 L 501 189.6 L 517 217.9 L 533 186.9 L 549 185.1 L 565 198.2" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 239.8 L 117 227.6 L 133 235.8 L 149 225.4 L 165 225 L 181 225.3 L 197 228.4 L 213 243.3 L 229 229.7 L 245 210.9 L 261 213.9 L 277 229.8 L 293 222.1 L 309 220.4 L 325 243 L 341 257.1 L 357 215.9 L 373 236.4 L 389 244.3 L 405 242 L 421 213.1 L 437 206.8 L 453 246.6 L 469 251.1 L 485 232.8 L 501 219.7 L 517 196.5 L 533 212.3 L 549 179.3 L 565 212.1" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 229.4 L 117 241.7 L 133 242.8 L 149 248.4 L 165 238.9 L 181 225.3 L 197 231.4 L 213 229.1 L 229 232.3 L 245 236.1 L 261 226.1 L 277 250.6 L 293 220.8 L 309 231.7 L 325 225.3 L 341 222.1 L 357 208.9 L 373 248 L 389 222.3 L 405 208.4 L 421 228.9 L 437 224.3 L 453 193.2 L 469 220.4 L 485 222.5 L 501 246.4 L 517 228.8 L 533 218.5 L 549 177.9 L 565 244.8" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 237.1 L 117 239.2 L 133 233.1 L 149 229.9 L 165 225.9 L 181 240.1 L 197 243.4 L 213 236.5 L 229 215.8 L 245 220.3 L 261 220.2 L 277 214.8 L 293 230.9 L 309 219.4 L 325 226.5 L 341 221.2 L 357 221.2 L 373 233.2 L 389 223.4 L 405 203.2 L 421 217.6 L 437 213.6 L 453 229.7 L 469 242.9 L 485 273.6 L 501 222.2 L 517 211.8 L 533 206.2 L 549 213.8 L 565 189.6" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 238.3 L 117 246.5 L 133 240.6 L 149 237.2 L 165 238.6 L 181 242.5 L 197 240.7 L 213 233.9 L 229 229.9 L 245 229.2 L 261 230 L 277 240.6 L 293 220.1 L 309 246.4 L 325 227.1 L 341 220.3 L 357 208.9 L 373 221.5 L 389 219.6 L 405 251 L 421 250.3 L 437 188.9 L 453 188.7 L 469 224.2 L 485 222.8 L 501 202.4 L 517 231.4 L 533 160.5 L 549 211.3 L 565 250.5" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 240.4 L 117 238.4 L 133 240.7 L 149 234.5 L 165 238.7 L 181 225.3 L 197 239.1 L 213 233.8 L 229 221.6 L 245 229.9 L 261 227.8 L 277 221.3 L 293 225.5 L 309 221.2 L 325 235.6 L 341 218.9 L 357 245.3 L 373 245.6 L 389 236 L 405 211.1 L 421 219.7 L 437 251.9 L 453 212 L 469 258.3 L 485 220.6 L 501 237.4 L 517 227.8 L 533 220.4 L 549 208.7 L 565 246.8" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 237.9 L 117 230.6 L 133 233 L 149 240.2 L 165 227.6 L 181 225.7 L 197 237.3 L 213 216.5 L 229 234.2 L 245 233.4 L 261 243.8 L 277 219.9 L 293 238.5 L 309 212.3 L 325 206.5 L 341 214.4 L 357 228.1 L 373 217.2 L 389 214.2 L 405 254.6 L 421 235.6 L 437 186.7 L 453 214.5 L 469 237.7 L 485 217.1 L 501 212.4 L 517 210.7 L 533 211.1 L 549 165.7 L 565 219.8" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 240.6 L 117 236.8 L 133 248.7 L 149 232.4 L 165 213.3 L 181 231.4 L 197 206.8 L 213 245 L 229 249.9 L 245 200.1 L 261 207.8 L 277 207.2 L 293 226.1 L 309 226.4 L 325 215.1 L 341 225.6 L 357 217 L 373 192.2 L 389 212.8 L 405 205.7 L 421 224.1 L 437 195.5 L 453 206.5 L 469 222.2 L 485 243.5 L 501 233.7 L 517 220.6 L 533 220.4 L 549 213.3 L 565 226.8" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 241.1 L 117 254 L 133 236.7 L 149 247.6 L 165 229 L 181 239.6 L 197 224.6 L 213 222.3 L 229 229.3 L 245 236.5 L 261 219.2 L 277 234.3 L 293 204.9 L 309 250.6 L 325 224 L 341 234.5 L 357 225 L 373 204.1 L 389 236.9 L 405 226.9 L 421 219.2 L 437 223.5 L 453 216.7 L 469 214.6 L 485 211.5 L 501 215.3 L 517 203.4 L 533 223.6 L 549 215.2 L 565 217.1" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 239.1 L 117 239.8 L 133 232.2 L 149 238.2 L 165 228.8 L 181 232.6 L 197 242 L 213 229.6 L 229 226.2 L 245 224.6 L 261 238.7 L 277 255.1 L 293 220.4 L 309 225.7 L 325 199.2 L 341 207.2 L 357 237.8 L 373 204.7 L 389 242.3 L 405 223.7 L 421 233.6 L 437 208.5 L 453 207.7 L 469 216.7 L 485 196.2 L 501 190.7 L 517 213.6 L 533 207 L 549 195.8 L 565 201.9" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 233.5 L 117 225 L 133 238.8 L 149 224.6 L 165 224.7 L 181 228 L 197 232.5 L 213 231.8 L 229 221.2 L 245 238.1 L 261 250.1 L 277 230.4 L 293 232.5 L 309 208 L 325 221.2 L 341 246.4 L 357 240.3 L 373 224.4 L 389 198 L 405 225.5 L 421 227.7 L 437 234.2 L 453 237.2 L 469 232.3 L 485 227.1 L 501 175.9 L 517 227.2 L 533 228.6 L 549 177.8 L 565 234.9" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 235.8 L 117 249.8 L 133 241.2 L 149 238.1 L 165 227.8 L 181 211.5 L 197 221.8 L 213 237.8 L 229 223.2 L 245 250.2 L 261 241.9 L 277 213.6 L 293 212 L 309 224.2 L 325 230.5 L 341 231.8 L 357 228.9 L 373 228.9 L 389 220.8 L 405 223.7 L 421 231 L 437 252.1 L 453 193.9 L 469 215.5 L 485 231.4 L 501 213.1 L 517 194.8 L 533 228.3 L 549 213.9 L 565 211.4" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 249.3 L 117 238.1 L 133 222.7 L 149 230.8 L 165 230.3 L 181 229.1 L 197 222.4 L 213 232.6 L 229 233.8 L 245 246.2 L 261 244 L 277 241.6 L 293 250.1 L 309 233.6 L 325 229.6 L 341 231 L 357 232.1 L 373 248.9 L 389 223.6 L 405 220.8 L 421 209.4 L 437 216 L 453 229.8 L 469 201.5 L 485 228.4 L 501 217.4 L 517 197.8 L 533 216.9 L 549 214.2 L 565 197.3" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 236.7 L 117 236.3 L 133 235.2 L 149 236 L 165 243.6 L 181 235.6 L 197 223.8 L 213 229.2 L 229 211.9 L 245 220.2 L 261 233.5 L 277 227.3 L 293 227.6 L 309 221.7 L 325 217.7 L 341 199 L 357 225.2 L 373 215.4 L 389 241.4 L 405 241.5 L 421 201.7 L 437 215.6 L 453 230.8 L 469 183 L 485 156.8 L 501 193.4 L 517 242.8 L 533 209 L 549 187.5 L 565 196.4" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 230.4 L 117 243.6 L 133 241.8 L 149 232.2 L 165 227.8 L 181 245.2 L 197 218.6 L 213 225.9 L 229 235.8 L 245 205.1 L 261 241.9 L 277 205.7 L 293 212.5 L 309 227.3 L 325 231 L 341 212.8 L 357 178.4 L 373 221.5 L 389 210.8 L 405 224.1 L 421 203.5 L 437 225.9 L 453 220.5 L 469 241.4 L 485 231.2 L 501 235.7 L 517 218 L 533 223.2 L 549 230.6 L 565 168.6" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 241.6 L 117 237.3 L 133 227.3 L 149 228.7 L 165 225.9 L 181 233.8 L 197 234.9 L 213 239.2 L 229 223.4 L 245 211.4 L 261 243 L 277 232.8 L 293 242.4 L 309 233.4 L 325 239.6 L 341 217.2 L 357 195.6 L 373 196.5 L 389 233.9 L 405 231.2 L 421 199.4 L 437 228.3 L 453 231.4 L 469 213.1 L 485 210.2 L 501 171 L 517 206.5 L 533 197.9 L 549 221.6 L 565 230.8" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 231.6 L 117 235.5 L 133 233.3 L 149 240.1 L 165 243.5 L 181 232.9 L 197 227.6 L 213 220.3 L 229 213.4 L 245 218.2 L 261 221.7 L 277 230.1 L 293 215 L 309 254.3 L 325 245 L 341 212.1 L 357 212 L 373 243.9 L 389 213.1 L 405 215.6 L 421 214.3 L 437 227.1 L 453 206.6 L 469 219.6 L 485 219.8 L 501 223.3 L 517 231.8 L 533 182.5 L 549 194.4 L 565 233.9" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 242.1 L 117 243.5 L 133 234.1 L 149 234.5 L 165 234.2 L 181 218.3 L 197 230.7 L 213 228 L 229 268.2 L 245 231 L 261 220.8 L 277 224.7 L 293 221 L 309 222.1 L 325 183.5 L 341 207.4 L 357 223.8 L 373 224.1 L 389 199 L 405 193.1 L 421 234.2 L 437 208 L 453 209.2 L 469 207.9 L 485 199.3 L 501 190 L 517 199.9 L 533 156.8 L 549 214.4 L 565 223.2" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 240 L 117 239.4 L 133 241.6 L 149 233.5 L 165 246.6 L 181 247 L 197 245.7 L 213 235 L 229 214.9 L 245 214.5 L 261 222.2 L 277 230.8 L 293 217.9 L 309 219.7 L 325 215.3 L 341 247.8 L 357 237.8 L 373 213.4 L 389 195.2 L 405 225.3 L 421 226 L 437 241.9 L 453 214.6 L 469 225.6 L 485 227.1 L 501 234.5 L 517 222.9 L 533 183 L 549 175.8 L 565 200.5" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 231.2 L 117 235.7 L 133 244.7 L 149 249.2 L 165 238.2 L 181 222.3 L 197 224.2 L 213 203.7 L 229 238.9 L 245 226.2 L 261 244.8 L 277 241.4 L 293 223.5 L 309 224.9 L 325 224.5 L 341 238.8 L 357 236.9 L 373 211.5 L 389 215.4 L 405 245.1 L 421 242.7 L 437 218.4 L 453 224.3 L 469 211.4 L 485 205 L 501 253.2 L 517 223 L 533 212.9 L 549 206.2 L 565 215.5" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 235.3 L 117 235.8 L 133 236.1 L 149 232.4 L 165 231.9 L 181 243.8 L 197 223.2 L 213 243.1 L 229 227 L 245 238.9 L 261 235.8 L 277 220.7 L 293 224.1 L 309 251.4 L 325 219 L 341 215.4 L 357 205.4 L 373 206.9 L 389 228.4 L 405 213 L 421 253 L 437 216 L 453 226.1 L 469 212.4 L 485 228.6 L 501 216.7 L 517 219.6 L 533 224.7 L 549 226.5 L 565 193.8" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 242 L 117 237.9 L 133 241.8 L 149 242.2 L 165 239.2 L 181 226.5 L 197 231.2 L 213 255.7 L 229 229.4 L 245 238.6 L 261 216.3 L 277 226.4 L 293 246.2 L 309 232.1 L 325 222.8 L 341 213.4 L 357 209.3 L 373 184.9 L 389 209.2 L 405 227.3 L 421 217.7 L 437 215.2 L 453 214.6 L 469 162.8 L 485 216 L 501 210.7 L 517 204.9 L 533 209.9 L 549 200 L 565 237.7" fill="none" stroke="#6E7B8B" stroke-width="0.8" stroke-linecap="round" stroke-linejoin="round" opacity="0.5"/>
<path d="M 85 240 L 101 238.9 L 117 237.8 L 133 236.7 L 149 235.6 L 165 234.5 L 181 233.4 L 197 232.3 L 213 231.2 L 229 230.1 L 245 229 L 261 227.9 L 277 226.8 L 293 225.7 L 309 224.6 L 325 223.5 L 341 222.4 L 357 221.3 L 373 220.2 L 389 219.1 L 405 218 L 421 216.9 L 437 215.8 L 453 214.7 L 469 213.6 L 485 212.5 L 501 211.4 L 517 210.3 L 533 209.2 L 549 208.1 L 565 207" fill="none" stroke="#1B3A5C" stroke-width="2.2" stroke-linecap="round" stroke-linejoin="round"/>
<text x="341" y="242.4" font-size="9" fill="#1B3A5C" font-weight="bold">均值路径</text>
<path d="M 558 125.2 L 558 288.8" fill="none" stroke="#E07B39" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ao36)" marker-start="url(#ao36)"/>
<text x="550" y="207" font-size="8.6" fill="#E07B39" text-anchor="end">95% 区间</text>
<rect x="615" y="105" width="140" height="64" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="685" y="129.6" font-size="11" fill="#1B3A5C" text-anchor="middle" font-weight="bold">路径模拟</text>
<text x="685" y="147.4" font-size="8.4" fill="#6E7B8B" text-anchor="middle">逆 CDF、固定种子</text>
<rect x="615" y="185" width="140" height="64" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="685" y="209.6" font-size="11" fill="#1B3A5C" text-anchor="middle" font-weight="bold">标准误 σ/√N</text>
<text x="685" y="227.4" font-size="8.4" fill="#6E7B8B" text-anchor="middle">95% 置信区间</text>
<rect x="615" y="265" width="140" height="64" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="685" y="289.6" font-size="11" fill="#1B3A5C" text-anchor="middle" font-weight="bold">方差缩减</text>
<text x="685" y="307.4" font-size="8.4" fill="#6E7B8B" text-anchor="middle">对偶 / 控制变量 / 分层</text>
<path d="M 685 169 L 685 185" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a36)"/>
<path d="M 685 249 L 685 265" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a36)"/>
<text x="75" y="344" font-size="9.4" fill="#6E7B8B">N = 10,000：SE = σ/√N，区间宽 ∝ 1/√N</text>
<text x="75" y="362" font-size="9.4" fill="#6E7B8B">对偶变量把方差减半 ≈ 路径数 ×4；控制变量要用已知期望</text>
<text x="75" y="380" font-size="9.4" fill="#1B3A5C">报告三件套：估计值、标准误、路径数（种子可复现）</text>
</svg>

> **图**：本章主线：路径均值只是一半，报出标准误和区间；对偶、控制变量把方差压下去。


In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



## 在量化研究中的位置

- **谁用**：衍生品定价、风险经理、组合研究、压力测试。
- **输入**：分布/SDE 参数、合约条款、持仓、置信水平。
- **输出**：价格/分布、VaR/ES、置信区间、收敛诊断。
- **在流程的哪一步**：解析解不可得时的定价与风险计算。
- **失败模式**：不报标准误、用真实漂移定价、路径数太少、忽略离散误差。
- **最小工作流**：定分布 → 生成随机数 → 模拟 → 算统计量/区间 → 方差缩减 → 校验。

---

## 1. 随机数：从均匀到正态

### 1.1 逆 CDF

若 $U\sim U(0,1)$，则

$$Z=\Phi^{-1}(U)\sim N(0,1).$$

**公式拆解**
- **符号**：$U$ 是 $[0,1)$ 上的均匀随机数，$\Phi$ 是标准正态分布的累积分布函数，$\Phi^{-1}$ 是它的分位数函数，$Z$ 是生成的标准正态变量，时间顺序是先生成 $U$ 再做变换。
- **结构**：把均匀的“概率刻度”映射成正态“分位数刻度”，因为 $\Phi$ 单调递增，变换后的 $Z$ 自动服从正态分布、中间密两端稀。
- **代入**：$U=0.975$ 得 $Z\approx1.96$、$U=0.5$ 得 0、$U=0.025$ 得约 $-1.96$，一次模拟抽 $n$ 个 $U$ 就得到 $n$ 条路径的冲击。
- **边界**：$U=0$ 或 1 时输出 $\pm\infty$、应先截断到 $[10^{-12},1-10^{-12}]$，双精度对极端分位数的分辨率也有限。
- **用途**：由均匀数生成任意分布（逆 CDF 法），是相关随机数与路径模拟的基础；它不保证随机数质量，报告要写清生成器、种子与路径数。

这是“逆 CDF 法”：把均匀随机数喂给目标分布的分位数函数。

### 1.2 可复现

每次模拟固定 `seed`；报告里写清随机数生成器/种子/路径数。否则别人无法复算。

### 1.3 代码：逆 CDF 与 GBM 路径



In [2]:

import numpy as np
from scipy.stats import norm

rng = np.random.default_rng(36)
u = rng.random(5)
z = norm.ppf(u)
print("U", np.round(u, 4))
print("Z", np.round(z, 4))

S0, mu, sigma, T, n_steps, n_paths = 100.0, 0.05, 0.20, 1.0, 252, 10_000
dt = T / n_steps
z_paths = rng.normal(size=(n_paths, n_steps))
S = S0 * np.exp(np.cumsum((mu - 0.5 * sigma ** 2) * dt + sigma * np.sqrt(dt) * z_paths, axis=1))
print("终值均值/标准差", round(S[:, -1].mean(), 3), round(S[:, -1].std(ddof=1), 3))


U [0.1806 0.3982 0.8938 0.4034 0.683 ]
Z [-0.913  -0.2579  1.2472 -0.2446  0.4761]
终值均值/标准差 105.107 21.248



**读数**：终值均值应接近 $S_0e^{\mu T}=105.13$；标准差随路径数波动。**模拟的均值本身也是随机变量，不是真值。**

**常见错误**：

1. 不固定种子，结果无法复现。
2. 用 `np.random.rand` 生成正态（应 `norm.ppf` 或 `normal`）。
3. 忘记 $-\frac12\sigma^2$ 漂移。

---

## 2. 误差：标准误与置信区间

### 2.1 标准误

若每条路径支付 $X_i$，均值 $\bar X$ 的标准误：

$$SE=\frac{s}{\sqrt n}.$$

**公式拆解**
- **符号**：$s$ 是单条路径结果（如期权支付）的样本标准差，$n$ 是独立路径数，$SE$ 是均值估计的标准误，单位与支付相同，它描述的是均值而非单条路径的波动。
- **结构**：$SE=s/\sqrt n$ 中分子是单条路径的噪声、分母是平均的降噪，路径数翻 4 倍标准误才减半，所以加路径很贵、方差缩减更值钱。
- **代入**：$s=10$ 元、$n=100$ 得 $SE=1$ 元，$n=400$ 得 0.5 元；本章价格 8.8、$SE\approx0.14$ 对应 95% 区间约 $\pm0.27$、总宽度约 0.55。
- **边界**：$n=1$ 等于没有平均，路径之间相关时公式会低估误差，小样本的区间要改用 t 分位数或 bootstrap。
- **用途**：报告蒙特卡洛精度、决定要跑多少条路径；不能拿“小数点后四位”代替置信区间。

95% 区间约 $\bar X\pm1.96SE$。

### 2.2 收敛速度

$$SE\propto\frac{1}{\sqrt n}.$$

路径数翻 4 倍，标准误才减半。**“多跑”很贵，方差缩减更值钱。**

### 2.3 代码：期权 MC



In [3]:

K, r = 100.0, 0.02
z_rn = rng.normal(size=(n_paths, n_steps))
S_rn = S0 * np.exp(np.cumsum((r - 0.5 * sigma ** 2) * dt + sigma * np.sqrt(dt) * z_rn, axis=1))
payoff = np.maximum(S_rn[:, -1] - K, 0) * np.exp(-r * T)
price, se = payoff.mean(), payoff.std(ddof=1) / np.sqrt(len(payoff))
print("MC 价格", round(price, 4), "标准误", round(se, 4))
print("95% 区间", (round(price - 1.96 * se, 3), round(price + 1.96 * se, 3)))


MC 价格 8.8073 标准误 0.1382
95% 区间 (np.float64(8.537), np.float64(9.078))



**读数**：价格约 8.8，标准误约 0.14；区间宽度约 0.55。**不报区间的“精确到小数点后四位”没有意义。**

**常见错误**：

1. 只报均值，不报 SE。
2. 用真实漂移 $\mu$ 给期权定价（应用 $r$）。
3. 路径数太少，区间跨 0 还下结论。

---

## 3. 方差缩减：同样路径数，更窄区间

### 3.1 对偶变量

用 $Z$ 和 $-Z$ 成对模拟，支付平均。若支付关于 $Z$ 单调，负相关，方差下降。

### 3.2 控制变量

选一个已知均值的变量 $C$（如 $S_T$），用回归系数 $\beta$ 调整：

$$X_{\text{cv}}=X-\beta(C-E[C]).$$

**公式拆解**
- **符号**：$X$ 是目标量（如期权支付），$C$ 是均值已知的控制变量（如到期价格），$\beta=\mathrm{Cov}(X,C)/\mathrm{Var}(C)$ 是回归系数，$X_{\text{cv}}$ 是调整后的估计量。
- **结构**：从 $X$ 中减去 $\beta(C-E[C])$，因为 $E[C-E[C]]=0$ 所以期望不变，而方差降为原来的 $1-\rho^2$ 倍、相关性越高降得越多。
- **代入**：$\beta=0.2$、样本内 $C$ 比真值高 0.4 时调整 $-0.08$，若相关系数 0.7，方差降到 0.51、标准误降约 29%。
- **边界**：$\beta=0$（不相关）时没有效果，$E[C]$ 估错会把误差一起搬进来，$\beta$ 必须在独立样本上估计。
- **用途**：路径数不变时缩窄模拟区间，是方差缩减里性价比最高的一招；它不改变期望，模型错了只会让错误更精确，也不能弥补模型本身的错误。

### 3.3 分层/重要性抽样

- 分层：把 $U$ 分成等概率层，每层抽固定数量；
- 重要性抽样：把抽样分布偏向“深度实值/尾部”，再乘似然比。

### 3.4 代码：对偶/控制变量



In [4]:

def mc_option(n, seed=3601, antithetic=False, control=False):
    rng = np.random.default_rng(seed)
    z = rng.normal(size=(n, n_steps))
    if antithetic:
        z = np.vstack([z, -z])
    S = S0 * np.exp(np.cumsum((r - 0.5 * sigma ** 2) * dt + sigma * np.sqrt(dt) * z, axis=1))
    x = np.maximum(S[:, -1] - K, 0) * np.exp(-r * T)
    if control:
        c = S[:, -1]
        beta = np.cov(x, c, ddof=1)[0, 1] / np.var(c, ddof=1)
        x = x - beta * (c - S0 * np.exp(r * T))
    return x.mean(), x.std(ddof=1) / np.sqrt(len(x))

for label, kwargs in [("普通", {}), ("对偶", {"antithetic": True}), ("控制变量", {"control": True}),
                      ("对偶+控制", {"antithetic": True, "control": True})]:
    m, s = mc_option(10_000, **kwargs)
    print(label, "价格", round(m, 4), "标准误", round(s, 4))


普通 价格 9.0687 标准误 0.1396


对偶 价格 8.8945 标准误 0.0979
控制变量 价格 8.8798 标准误 0.0589


对偶+控制

 价格 8.8954 标准误 0.0417



**读数**：对偶/控制变量都能把标准误降下来；组合使用通常最好。**方差缩减不改变期望，只改变误差。**

**常见错误**：

1. 对偶变量不是 $Z$ 与 $-Z$。
2. 控制变量的均值写成样本均值（应已知）。
3. 用了方差缩减却不比较 SE。

---

## 4. 应用：VaR、ES 与组合分布

### 4.1 VaR/ES

- VaR$_{95\%}$：损失分布的 95% 分位；
- ES$_{95\%}$：超过 VaR 的平均损失。

用模拟样本直接取分位数/条件均值。

### 4.2 代码：厚尾收益的 VaR/ES



In [5]:

rng = np.random.default_rng(3602)
ret = rng.standard_t(df=5, size=200_000) * 0.01
var95 = -np.quantile(ret, 0.05)
es95 = -ret[ret <= -var95].mean()
print("VaR95", round(var95, 4), "ES95", round(es95, 4))


VaR95 0.0202 ES95 0.0292



**读数**：厚尾下 ES 明显大于 VaR；只报 VaR 会低估尾部。**模拟 VaR 的误差也要用 SE/区间，不是只看一个数。**

**常见错误**：

1. 把 VaR 当最大损失。
2. 用正态模拟厚尾。
3. 不报样本量/置信区间。

---

## 5. 拟蒙特卡洛：用低差异序列

### 5.1 思路

普通 MC 的均匀点会成簇；Sobol/Halton 等低差异序列更均匀，收敛更快（但误差估计更复杂）。

### 5.2 代码：Sobol



In [6]:

from scipy.stats import qmc

sobol = qmc.Sobol(d=1, scramble=True, seed=36)
u_qmc = sobol.random(8192).ravel()
z_qmc = norm.ppf(np.clip(u_qmc, 1e-12, 1 - 1e-12))
S_T_qmc = S0 * np.exp((r - 0.5 * sigma ** 2) * T + sigma * np.sqrt(T) * z_qmc)
pay_qmc = np.maximum(S_T_qmc - K, 0) * np.exp(-r * T)
print("QMC 价格", round(pay_qmc.mean(), 4), "路径数", len(pay_qmc))


QMC 价格 8.9162 路径数 8192



**读数**：QMC 用更少的点给出接近的价格；但它假设被积函数足够光滑，误差估计不能用普通 $s/\sqrt n$。

**常见错误**：

1. 把 QMC 当“无误差”。
2. 用普通 SE 公式估 QMC 误差。
3. 维度很高时盲目用 QMC。

---

## 本章小结

### 一张表

| 概念 | 公式/做法 | 用途 |
|---|---|---|
| 逆 CDF | $Z=\Phi^{-1}(U)$ | 生成分布 |
| MC 标准误 | $s/\sqrt n$ | 误差 |
| 对偶 | $Z,-Z$ | 降方差 |
| 控制变量 | $X-\beta(C-E C)$ | 降方差 |
| VaR/ES | 分位/条件均值 | 尾部风险 |
| QMC | Sobol/Halton | 更快收敛 |

### 八句话

1. 模拟均值也是随机变量。
2. 不报 SE 等于没报结果。
3. 路径数翻 4 倍，SE 才减半。
4. 对偶/控制变量能降方差。
5. 期权定价用 $r$，不用 $\mu$。
6. VaR 不是最大损失。
7. ES 比 VaR 更看尾部。
8. QMC 不是“无误差”。

---

## 本章习题

**1.** $U=0.975$，$\Phi^{-1}(U)$ 约多少？

**2.** 10,000 条路径，支付标准差 15，SE 多少？

**3.** 要让 SE 减半，路径数变多少？

**4.** 对偶变量为什么能降方差？

**5.** 期权 MC 用 $\mu$ 还是 $r$？为什么？

**6.** VaR95=2%，ES95=3.5%，说明什么？

**7.** QMC 的误差能不能用 $s/\sqrt n$？

**8.** 为什么“价格 8.8321”不一定是好消息？

<details><summary>参考答案</summary>

**1.** 约 1.96。

**2.** $15/\sqrt{10000}=0.15$。

**3.** 4 倍（40,000）。

**4.** $Z$ 与 $-Z$ 的支付负相关，平均后波动下降。

**5.** 用 $r$；风险中性定价。

**6.** 尾部平均损失比 VaR 大，分布厚尾。

**7.** 不能；QMC 误差估计不同。

**8.** 可能标准误很大；没有区间的精度是假精度。

</details>

---

## 参考资料

- Glasserman, P. *Monte Carlo Methods in Financial Engineering*。
- Kloeden, P. E. & Platen, E. *Numerical Solution of Stochastic Differential Equations*。
